In [1]:
import time
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report


In [2]:
df = pd.read_csv("C:\\Users\\aeaea\\OneDrive\\سطح المكتب\\ml_features_and_labels.csv")
df.head()

manifest = pd.read_csv("C:\\Users\\aeaea\\OneDrive\\سطح المكتب\\scenario_manifest.csv")

df['merge_id'] = df['ID'].astype(str).str.replace('.pcap', '', regex=False)
manifest['merge_id'] = manifest['ID'].astype(str).str.replace('.pcap', '', regex=False)

manifest_features = manifest[['merge_id', 'taxonomy_class', 'run_idx']].copy()

df = df.merge(manifest_features, on='merge_id', how='left')

df = df.drop(columns=['merge_id'])

df.head()

,e2c_total_bytes,e4_entropy_h,e5_entropy_c,e6_time_char,e6b_flow_duration_ms,e2_client_size,e2_client_record_len,e2_server_record_len,e3_cert_parsed,e1_alg_suite_Unknown(0x11eb),...,e2b_tls_version_TLS1.0,e2b_ciphersuite_2,e2b_ciphersuite_54,e2b_ciphersuite_60,label,split,taxonomy,ID,taxonomy_class,run_idx
0,8182,4.7889,5.8077,2.45,149.827957,32.0,285,5316.0,False,False,...,True,False,False,True,1,train,eval_test_network,net_high_jitter_classic_run1,Network,1
1,8126,5.0000,5.8367,2.44,231.020927,32.0,285,5316.0,False,False,...,True,False,False,True,1,val,eval_test_network,net_high_jitter_classic_run10,Network,10
2,8272,4.7889,5.9056,2.73,292.967796,32.0,285,5316.0,False,False,...,True,False,False,True,1,train,eval_test_network,net_high_jitter_classic_run100,Network,100
3,8272,4.8125,5.9056,2.31,156.050205,32.0,285,5316.0,False,False,...,True,False,False,True,1,test,eval_test_network,net_high_jitter_classic_run101,Network,101
4,8272,4.8125,5.8367,4.45,193.972111,32.0,285,5316.0,False,False,...,True,False,False,True,1,test,eval_test_network,net_high_jitter_classic_run102,Network,102


In [3]:
feature_columns = [col for col in df.columns if col not in ['label', 'split', 'taxonomy', 'ID']]
X = df[feature_columns].replace({True: 1, False: 0, 'TRUE': 1, 'FALSE': 0})
X = X.apply(pd.to_numeric, errors='coerce').fillna(0)
y = df['label']

X_train = X[df['split'] == 'train']
y_train = y[df['split'] == 'train']
X_val = X[df['split'] == 'val']
y_val = y[df['split'] == 'val']
X_test = X[df['split'] == 'test']
y_test = y[df['split'] == 'test']

C:\Users\aeaea\AppData\Local\Temp\ipykernel_14604\2147704158.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X = df[feature_columns].replace({True: 1, False: 0, 'TRUE': 1, 'FALSE': 0})


In [ ]:
model = CatBoostClassifier(iterations=400, learning_rate=0.1, depth=6, random_seed=42, verbose=0)
# iterations doesn't make much of a difference
start = time.perf_counter()

# Training
model.fit(X_train, y_train)

end = time.perf_counter()
training_time = end - start
print(f'Training time: {training_time:.2f} seconds')

Training time: 6.04 seconds


In [7]:
start = time.perf_counter()
y_val_pred = model.predict(X_val).ravel()
y_test_pred = model.predict(X_test).ravel()
end = time.perf_counter()
testing_time = end - start

results = pd.DataFrame({
    'Split': ['Validation', 'Test'],
    'Accuracy': [accuracy_score(y_val, y_val_pred), accuracy_score(y_test, y_test_pred)],
    'Precision': [precision_score(y_val, y_val_pred, zero_division=0), precision_score(y_test, y_test_pred, zero_division=0)],
    'Recall': [recall_score(y_val, y_val_pred, zero_division=0), recall_score(y_test, y_test_pred, zero_division=0)],
    'F1 Score': [f1_score(y_val, y_val_pred, zero_division=0), f1_score(y_test, y_test_pred, zero_division=0)]
})

print(f'Testing time: {testing_time:.2f} seconds')
print('\nTest classification report:')
print(classification_report(y_test, y_test_pred, zero_division=0))
results

Testing time: 0.03 seconds

Test classification report:
              precision    recall  f1-score   support

           0       0.99      0.98      0.98      7000
           1       0.89      0.90      0.90      1002

    accuracy                           0.97      8002
   macro avg       0.94      0.94      0.94      8002
weighted avg       0.97      0.97      0.97      8002



,Split,Accuracy,Precision,Recall,F1 Score
0,Validation,0.975381,0.892683,0.913174,0.902812
1,Test,0.973757,0.888998,0.903194,0.896040
